# 04 — LLM Evaluation

Ecommerce KPI RAG Capstone — LLM Zoomcamp 2026

We compare two prompt strategies for the generation step: a **minimal prompt** (just context +
question) against a **structured prompt** (explicit instructions to cite numbers, flag missing
data, and keep answers concise). Each is scored by an LLM-as-judge on faithfulness (does the
answer only use numbers present in the context?) and relevance.

## 1. Setup

In [4]:
import os
import sys
import json

sys.path.insert(0, "../src")
from rag_index import text_search

from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

with open("../data/eval_questions.json") as f:
    eval_questions = json.load(f)

with open("../data/documents.json") as f:
    documents = json.load(f)

# use a sample of 15 questions to keep API usage light
import random
random.seed(42)
sample_questions = random.sample(eval_questions, 15)

## 2. Two prompt strategies

In [5]:
PROMPT_MINIMAL = """Context:
{context}

Question: {question}
Answer:"""

PROMPT_STRUCTURED = """You are an ecommerce analytics assistant. Answer the QUESTION using
only the numbers in CONTEXT below. Rules:
- Quote exact figures from the context — never estimate or round differently than the source.
- If the context does not contain the answer, say "I don't have that data" instead of guessing.
- Keep the answer to 1-2 sentences.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

def build_context(results):
    return "\n".join(f"- {r['text']}" for r in results)

def generate(prompt_template, question, num_results=5):
    results = text_search(question, num_results=num_results)
    context = build_context(results)
    prompt = prompt_template.format(context=context, question=question)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="low",
    )
    return response.choices[0].message.content

## 3. Generate answers with both prompts

In [6]:
results = []
for gt in sample_questions:
    q = gt["question"]
    answer_minimal = generate(PROMPT_MINIMAL, q)
    answer_structured = generate(PROMPT_STRUCTURED, q)
    results.append({
        "question": q,
        "doc_id": gt["doc_id"],
        "answer_minimal": answer_minimal,
        "answer_structured": answer_structured,
    })

results[0]

{'question': 'What is the total revenue in BR?',
 'doc_id': 7,
 'answer_minimal': 'The total revenue in Brazil (BR) is **$836,797.43**.',
 'answer_structured': 'The total revenue in Brazil (BR) is **$836,797.43**.'}

## 4. LLM-as-judge

We ask the same model to score each answer 1-5 on two dimensions:
- **Faithfulness** — does the answer only use numbers that actually appear in the source document?
- **Relevance** — does the answer actually address the question asked?

In [7]:
JUDGE_PROMPT = """You are evaluating an AI-generated answer to a question about ecommerce KPIs.

SOURCE DOCUMENT: {source_text}
QUESTION: {question}
ANSWER: {answer}

Rate the answer from 1 (poor) to 5 (excellent) on two dimensions. Respond with ONLY a JSON
object, no other text:
{{"faithfulness": <1-5>, "relevance": <1-5>}}

faithfulness = does the answer only use numbers that actually appear in the source document?
relevance = does the answer actually address the question asked?"""

def judge(question, answer, doc_id):
    source_text = next(d["text"] for d in documents if d["id"] == doc_id)
    prompt = JUDGE_PROMPT.format(source_text=source_text, question=question, answer=answer)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="low",
    )
    try:
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        return {"faithfulness": None, "relevance": None}

In [9]:
for r in results:
    r["judge_minimal"] = judge(r["question"], r["answer_minimal"], r["doc_id"])
    r["judge_structured"] = judge(r["question"], r["answer_structured"], r["doc_id"])

with open("../data/llm_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

## 5. Compare scores

In [10]:
import pandas as pd

rows = []
for r in results:
    rows.append({
        "prompt": "minimal",
        "faithfulness": r["judge_minimal"]["faithfulness"],
        "relevance": r["judge_minimal"]["relevance"],
    })
    rows.append({
        "prompt": "structured",
        "faithfulness": r["judge_structured"]["faithfulness"],
        "relevance": r["judge_structured"]["relevance"],
    })

scores = pd.DataFrame(rows)
scores.groupby("prompt")[["faithfulness", "relevance"]].mean()

,faithfulness,relevance
prompt,,
minimal,5.0,5.0
structured,5.0,5.0


## 6. Record results

Fill in the actual scores after running Sections 3-5 locally, and copy this table into the README.

| Prompt | Avg. Faithfulness | Avg. Relevance |
|---|-------------------|----------------|
| Minimal | 5.0               | 5.0            |
| Structured | 5.0               | 5.0            |

**Expectation going in:** the structured prompt explicitly instructs the model to only quote
numbers present in the context and to admit when it doesn't know — this should measurably
improve faithfulness (fewer fabricated or rounded-differently figures) at little cost to
relevance, since both prompts retrieve the same context.